# Dataing Demo Notebook

This notebook demonstrates the Dataing SDK for data quality investigation.

## Getting Started

**GUI-First Approach (Recommended):**
1. Look at the **Dataing sidebar** on the left (click the magnifying glass icon if not visible)
2. Click **Connect** to configure your backend connection
3. The sidebar will show your connection status and let you attach to datasources

**Magic Commands (Alternative):**
You can also use magic commands directly in code cells - see examples below.

## Magic Commands Reference

The `%dataing` magic provides these commands:
- `%dataing connect` - Connect to backend (uses GUI config if available)
- `%dataing status` - Show current connection and binding status
- `%dataing attach <asset>` - Attach to a datasource
- `%dataing lineage [--depth N] [--direction DIR] [--export FILE]` - Visualize data lineage
- `%dataing ask <question>` - Start an investigation

In [ ]:
# Load the Dataing magic extension
%load_ext dataing_notebook

In [ ]:
# Check current status
# If you connected via the sidebar, this will show your connection
%dataing status

In [ ]:
# Connect to backend (if not already connected via sidebar)
# Without arguments, uses GUI configuration if available
%dataing connect

## Attaching to a Datasource

Use the sidebar to search and attach to a datasource, or use the magic command:

In [ ]:
# Example: Attach to the demo DuckDB datasource
%dataing attach duckdb://demo/fixtures

## Running an Investigation

Once connected and attached to a datasource, you can ask questions about your data:

In [ ]:
# Example investigation question
%dataing ask 'Why did null values spike in the orders table last week?'

## Lineage Visualization

Dataing provides interactive lineage graph visualization to explore data dependencies.
Once attached to a datasource with lineage information, use `%dataing lineage` to visualize:

**Options:**
- `--depth N` or `-d N` - Control traversal depth (1-10, default: 2)
- `--direction upstream|downstream|both` - Filter by direction (default: both)
- `--export FILE` or `-e FILE` - Export graph to PNG or SVG file

In [ ]:
# View lineage graph with default settings (depth=2, direction=both)
%dataing lineage

In [ ]:
# View only upstream dependencies (what feeds into this table)
%dataing lineage --direction upstream

In [ ]:
# View downstream consumers with deeper traversal
%dataing lineage --depth 5 --direction downstream

In [ ]:
# Export graph to PNG for sharing
%dataing lineage --export lineage_graph.png

### Programmatic Lineage Visualization

You can also render lineage graphs directly using the `render_lineage_graph` function.
This is useful for custom integrations or when working with lineage data from other sources:

In [ ]:
from dataing_notebook.lineage_graph import render_lineage_graph

# Sample e-commerce data lineage
sample_lineage = {
    "root": "analytics.order_summary",
    "datasets": {
        "raw.orders": {"type": "source", "platform": "postgres", "description": "Raw order events from production"},
        "raw.customers": {"type": "source", "platform": "postgres", "description": "Customer master data"},
        "raw.products": {"type": "source", "platform": "postgres", "description": "Product catalog"},
        "staging.orders": {"type": "table", "platform": "duckdb", "description": "Cleaned and validated orders"},
        "staging.customers": {"type": "table", "platform": "duckdb", "description": "Deduplicated customers"},
        "analytics.order_summary": {"type": "view", "platform": "duckdb", "description": "Daily order aggregations",
                                     "owners": ["data-team"], "tags": ["production", "dashboard"]},
        "analytics.customer_ltv": {"type": "view", "platform": "duckdb", "description": "Customer lifetime value"},
    },
    "edges": [
        {"source": "raw.orders", "target": "staging.orders", "edge_type": "transforms"},
        {"source": "raw.customers", "target": "staging.customers", "edge_type": "transforms"},
        {"source": "staging.orders", "target": "analytics.order_summary", "edge_type": "transforms"},
        {"source": "staging.customers", "target": "analytics.order_summary", "edge_type": "joins"},
        {"source": "raw.products", "target": "analytics.order_summary", "edge_type": "joins"},
        {"source": "analytics.order_summary", "target": "analytics.customer_ltv", "edge_type": "transforms"},
    ],
    "jobs": {},
}

# Render the interactive graph (click nodes to see details!)
render_lineage_graph(sample_lineage, depth=3, direction="both")

In [ ]:
# View only upstream dependencies (what feeds into the root table)
render_lineage_graph(sample_lineage, depth=2, direction="upstream")

In [ ]:
# View downstream consumers (what depends on the root table)
render_lineage_graph(sample_lineage, depth=2, direction="downstream")

In [ ]:
# Export to PNG for sharing (uncomment to run)
# from dataing_notebook.lineage_graph import export_lineage_graph
# export_lineage_graph(sample_lineage, "lineage_demo.png", fmt="png", depth=3)
# print("Exported to lineage_demo.png")

## Using the SDK Directly

For programmatic access, you can use the Dataing SDK:

In [ ]:
from dataing_sdk import DataingClient

# Create a client (uses environment variables or pass explicitly)
# client = DataingClient(base_url="http://localhost:8000", api_key="dd_demo_12345")

# List datasources
# datasources = client.datasources.list()
# print(datasources)

## Demo Fixtures

The demo includes pre-built anomaly scenarios:
- `null_spike` - Sudden increase in NULL values
- `volume_drop` - Unexpected drop in row count
- `schema_drift` - Column type changes
- `duplicates` - Duplicate primary keys
- `late_arriving` - Data arriving late
- `orphaned_records` - Foreign key violations